# 05. Optimización de hiperparámetros de modelos IT

Este notebook realiza la fase de optimización de hiperparámetros de los modelos seleccionados tras la primera criba:

- CatBoost
- Gradient Boosting
- LightGBM
- XGBoost
- Extra Trees

La optimización utiliza exclusivamente los conjuntos **Train** y **Validation**. El conjunto **Test permanece completamente aislado** y no se utiliza para seleccionar hiperparámetros ni modelos.

La búsqueda se realiza mediante muestreo aleatorio de configuraciones (`ParameterSampler`) y una validación fija temporal:

- Train: `ID_Semana` 1–91
- Validation: `ID_Semana` 92–111
- Test: `ID_Semana` 112–131

Cada configuración se cronometra individualmente y los resultados se guardan después de cada ejecución para evitar perder progreso si el proceso se interrumpe.

In [1]:
# ============================================================
# 1. IMPORTACIONES Y CONFIGURACIÓN
# ============================================================

import os
import time
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Número de configuraciones por modelo.
# Se puede aumentar posteriormente si el tiempo disponible lo permite.
N_ITER_POR_MODELO = 6

print("=" * 70)
print("CONFIGURACIÓN DE OPTIMIZACIÓN")
print("=" * 70)
print(f"Configuraciones por modelo: {N_ITER_POR_MODELO}")
print("Semilla aleatoria:", RANDOM_STATE)

CONFIGURACIÓN DE OPTIMIZACIÓN
Configuraciones por modelo: 6
Semilla aleatoria: 42


### Criterio de búsqueda

Se utiliza una búsqueda aleatoria controlada en lugar de una búsqueda exhaustiva. Con el tamaño de la base de datos, una búsqueda en rejilla podría generar un número excesivo de entrenamientos.

El número de configuraciones se concentra en los hiperparámetros con mayor impacto sobre la complejidad, capacidad de aprendizaje y generalización de cada algoritmo.

El criterio principal de comparación será el **R² promedio de los cuatro horizontes**, complementado con MAE y RMSE.

In [2]:
# ============================================================
# 2. CARGA DE LA TABLA DE MODELADO
# ============================================================

# Buscar el archivo en la carpeta actual o en la carpeta Resultados/Datos
posibles_rutas = [
    os.path.join(os.getcwd(), "Tabla_Entrenamiento_IT_Modelado.csv"),
    os.path.join(os.getcwd(), "Datos", "Tabla_Entrenamiento_IT_Modelado.csv"),
    os.path.join(os.getcwd(), "..", "Tabla_Entrenamiento_IT_Modelado.csv")
]

ruta_datos = next(
    (p for p in posibles_rutas if os.path.exists(p)),
    None
)

if ruta_datos is None:
    raise FileNotFoundError(
        "No se encontró Tabla_Entrenamiento_IT_Modelado.csv. "
        "Coloca el archivo en la carpeta del notebook o ajusta 'ruta_datos'."
    )

df = pd.read_csv(
    ruta_datos,
    sep=";",
    encoding="utf-8-sig"
)

print("Archivo:", ruta_datos)
print("Filas:", len(df))
print("Columnas:", len(df.columns))

Archivo: c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Tabla_Entrenamiento_IT_Modelado.csv
Filas: 1755108
Columnas: 40


In [3]:
# ============================================================
# 3. DEFINICIÓN DE PREDICTORES Y OBJETIVOS
# ============================================================

target_names = [
    "IT_Futura_W1",
    "IT_Futura_W2",
    "IT_Futura_W3",
    "IT_Futura_W4"
]

excluded_columns = [
    "CodEmpleado",
    "FechaInicio",
    "FechaFin",
    "IdAlta",
    "IdContrato",
    "UltimoStatus",
    "FechaInicioAlta",
    "FechaFinAlta",
    "ParticipacionDesde",
    "ParticipacionHasta",
    "OrganizacionDesdeAsignada",
    "OrganizacionHastaAsignada",
    "OrigenAsignacionOrganizativa",
    "Nacimiento",
    "ITAbierto",
    "ElegiblePrediccion"
]

categorical_features = [
    "ClaseContrato",
    "FamiliaFuncion",
    "Centro",
    "UnidadOrganizativa",
    "Funcion",
    "Area",
    "Servicio",
    "Sexo",
    "TipoJornada"
]

predictor_columns = [
    c for c in df.columns
    if c not in excluded_columns + target_names
]

numeric_features = [
    c for c in predictor_columns
    if c not in categorical_features
]

X = df[predictor_columns].copy()
y = df[target_names].copy()

print("Predictores:", len(predictor_columns))
print("Numéricas:", len(numeric_features))
print("Categóricas:", len(categorical_features))
print("Objetivos:", len(target_names))

Predictores: 36
Numéricas: 27
Categóricas: 9
Objetivos: 4


In [5]:
# ============================================================
# 4. PARTICIÓN TEMPORAL
# ============================================================

train_mask = df["ID_Semana"] <= 91
val_mask = (
    (df["ID_Semana"] >= 92) &
    (df["ID_Semana"] <= 111)
)
test_mask = df["ID_Semana"] >= 112

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

print("=" * 70)
print("PARTICIÓN TEMPORAL")
print("=" * 70)

print(f"Train:      {len(X_train):,} filas | semanas 1-91")
print(f"Validation: {len(X_val):,} filas | semanas 92-111")
print(f"Test:       {len(X_test):,} filas | semanas 112-131")

print("⚠️ TEST PERMANECE AISLADO.")

PARTICIÓN TEMPORAL
Train:      1,206,215 filas | semanas 1-91
Validation: 273,238 filas | semanas 92-111
Test:       275,655 filas | semanas 112-131
⚠️ TEST PERMANECE AISLADO.


In [6]:
# ============================================================
# 5. PREPROCESADORES
# ============================================================

# Para modelos sklearn/XGBoost/LightGBM:
# categóricas -> One-Hot Encoding
# numéricas -> imputación por media
# categóricas -> imputación por moda + One-Hot

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor_tree = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocesador preparado.")

Preprocesador preparado.


In [7]:
# ============================================================
# 6. FUNCIONES DE EVALUACIÓN Y REGISTRO
# ============================================================

def evaluar_predicciones(y_real, y_pred, modelo, trial, tiempo_s):
    filas = []

    for i, objetivo in enumerate(target_names):
        real = np.asarray(y_real.iloc[:, i])
        pred = np.asarray(y_pred[:, i])

        mae = mean_absolute_error(real, pred)
        rmse = np.sqrt(mean_squared_error(real, pred))
        r2 = r2_score(real, pred)

        filas.append({
            "Modelo": modelo,
            "Trial": trial,
            "Conjunto": "Validation",
            "Objetivo": objetivo,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Tiempo_entrenamiento_s": tiempo_s
        })

    detalle = pd.DataFrame(filas)

    resumen = {
        "Modelo": modelo,
        "Trial": trial,
        "MAE_Promedio": detalle["MAE"].mean(),
        "RMSE_Promedio": detalle["RMSE"].mean(),
        "R2_Promedio": detalle["R2"].mean(),
        "Tiempo_entrenamiento_s": tiempo_s
    }

    return detalle, resumen


def guardar_progreso():
    os.makedirs(carpeta_resultados, exist_ok=True)

    resultados_optimizacion_df.to_csv(
        ruta_resumen,
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    detalle_optimizacion_df.to_csv(
        ruta_detalle,
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

In [8]:
# ============================================================
# 7. CARPETA Y ARCHIVOS DE PROGRESO
# ============================================================

carpeta_resultados = os.path.join(
    os.getcwd(),
    "Resultados_Optimizacion"
)

os.makedirs(
    carpeta_resultados,
    exist_ok=True
)

ruta_resumen = os.path.join(
    carpeta_resultados,
    "Resultados_Optimizacion_IT.csv"
)

ruta_detalle = os.path.join(
    carpeta_resultados,
    "Resultados_Optimizacion_IT_Detalle.csv"
)

# Recuperar resultados previos si el notebook se interrumpió
if os.path.exists(ruta_resumen):
    resultados_optimizacion_df = pd.read_csv(
        ruta_resumen,
        sep=";",
        encoding="utf-8-sig"
    )
else:
    resultados_optimizacion_df = pd.DataFrame()

if os.path.exists(ruta_detalle):
    detalle_optimizacion_df = pd.read_csv(
        ruta_detalle,
        sep=";",
        encoding="utf-8-sig"
    )
else:
    detalle_optimizacion_df = pd.DataFrame()

print("Carpeta de resultados:", carpeta_resultados)
print("Trials previos recuperados:", len(resultados_optimizacion_df))

Carpeta de resultados: c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados_Optimizacion
Trials previos recuperados: 0


## 8. Espacios de búsqueda

Los espacios de búsqueda se han definido de forma deliberadamente contenida para que la optimización sea viable con 1,2 millones de observaciones de entrenamiento.

Los rangos no representan una búsqueda exhaustiva: se muestrean aleatoriamente `N_ITER_POR_MODELO` configuraciones por algoritmo.

In [9]:
# ============================================================
# 8. ESPACIOS DE BÚSQUEDA
# ============================================================

param_distributions = {

    "Gradient Boosting": {
        "model__estimator__n_estimators": [100, 150, 200, 300],
        "model__estimator__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.10],
        "model__estimator__max_depth": [2, 3, 4, 5],
        "model__estimator__min_samples_leaf": [10, 20, 50, 100],
        "model__estimator__subsample": [0.7, 0.8, 0.9, 1.0]
    },

    "Extra Trees": {
        "model__n_estimators": [100, 150, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__max_features": [0.5, 0.7, 1.0]
    },

    "XGBoost": {
        "model__n_estimators": [100, 200, 300, 400],
        "model__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.10],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__min_child_weight": [1, 3, 5, 10],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
    },

    "LightGBM": {
        "model__estimator__n_estimators": [100, 200, 300, 400],
        "model__estimator__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.10],
        "model__estimator__num_leaves": [15, 31, 63, 127],
        "model__estimator__max_depth": [-1, 5, 10, 15],
        "model__estimator__min_child_samples": [20, 50, 100],
        "model__estimator__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__estimator__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
    }
}

# CatBoost utiliza nombres de hiperparámetros diferentes
param_distributions_catboost = {
    "iterations": [200, 300, 400, 500, 700],
    "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.10],
    "depth": [5, 6, 7, 8, 9, 10],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "random_strength": [0, 0.5, 1, 2]
}

print("Espacios de búsqueda definidos.")

Espacios de búsqueda definidos.


In [10]:
# ============================================================
# 9. FUNCIONES PARA CONSTRUIR MODELOS
# ============================================================

def construir_modelo(modelo, params=None):

    if modelo == "Gradient Boosting":

        base = GradientBoostingRegressor(
            random_state=RANDOM_STATE
        )

        return Pipeline([
            ("preprocess", preprocessor_tree),
            ("model", MultiOutputRegressor(
                base,
                n_jobs=-1
            ))
        ])

    if modelo == "Extra Trees":

        base = ExtraTreesRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        return Pipeline([
            ("preprocess", preprocessor_tree),
            ("model", base)
        ])

    if modelo == "XGBoost":

        base = XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist"
        )

        return Pipeline([
            ("preprocess", preprocessor_tree),
            ("model", base)
        ])

    if modelo == "LightGBM":

        base = LGBMRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1
        )

        return Pipeline([
            ("preprocess", preprocessor_tree),
            ("model", MultiOutputRegressor(
                base,
                n_jobs=-1
            ))
        ])

    raise ValueError(f"Modelo no reconocido: {modelo}")

In [12]:
# ============================================================
# 10. OPTIMIZACIÓN: MODELOS SKLEARN / XGBOOST / LIGHTGBM
# ============================================================

modelos_a_optimizar = [
    "Gradient Boosting",
    "Extra Trees",
    "XGBoost",
    "LightGBM"
]

for modelo in modelos_a_optimizar:

    print("\n" + "=" * 70)
    print(f"OPTIMIZANDO: {modelo}")
    print("=" * 70)

    # Generar configuraciones reproducibles
    configuraciones = list(
        ParameterSampler(
            param_distributions[modelo],
            n_iter=N_ITER_POR_MODELO,
            random_state=RANDOM_STATE
        )
    )

    for trial_idx, params in enumerate(configuraciones, start=1):

        # Comprobar si este trial ya fue ejecutado
        ya_ejecutado = (
            (len(resultados_optimizacion_df) > 0) and
            (resultados_optimizacion_df["Modelo"] == modelo).any() and
            (resultados_optimizacion_df["Trial"] == trial_idx).any()
        )

        if ya_ejecutado:
            print(
                f"Trial {trial_idx}/{len(configuraciones)} "
                f"ya ejecutado. Se omite."
            )
            continue

        print("\n" + "-" * 70)
        print(
            f"{modelo} | "
            f"Trial {trial_idx}/{len(configuraciones)}"
        )
        print("Hiperparámetros:")
        print(params)

        modelo_pipeline = construir_modelo(modelo)

        inicio = time.time()

        try:

            modelo_pipeline.set_params(**params)

            modelo_pipeline.fit(
                X_train,
                y_train
            )

            tiempo_s = time.time() - inicio

            y_pred = modelo_pipeline.predict(
                X_val
            )

            detalle, resumen = evaluar_predicciones(
                y_val,
                y_pred,
                modelo,
                trial_idx,
                tiempo_s
            )

            # Guardar hiperparámetros como texto JSON
            resumen["Hiperparametros"] = json.dumps(
                params,
                ensure_ascii=False,
                default=str
            )

            detalle["Hiperparametros"] = resumen["Hiperparametros"]

            resultados_optimizacion_df = pd.concat(
                [
                    resultados_optimizacion_df,
                    pd.DataFrame([resumen])
                ],
                ignore_index=True
            )

            detalle_optimizacion_df = pd.concat(
                [
                    detalle_optimizacion_df,
                    detalle
                ],
                ignore_index=True
            )

            guardar_progreso()

            print(
                f"✓ Completado en "
                f"{tiempo_s / 60:.2f} min"
            )
            print(
                f"MAE={resumen['MAE_Promedio']:.4f} | "
                f"RMSE={resumen['RMSE_Promedio']:.4f} | "
                f"R²={resumen['R2_Promedio']:.4f}"
            )

        except Exception as error:

            tiempo_s = time.time() - inicio

            print(
                f"⚠️ ERROR en {modelo}, "
                f"trial {trial_idx}:"
            )
            print(error)

            error_row = {
                "Modelo": modelo,
                "Trial": trial_idx,
                "MAE_Promedio": np.nan,
                "RMSE_Promedio": np.nan,
                "R2_Promedio": np.nan,
                "Tiempo_entrenamiento_s": tiempo_s,
                "Hiperparametros": json.dumps(
                    params,
                    ensure_ascii=False,
                    default=str
                ),
                "Estado": f"ERROR: {str(error)}"
            }

            resultados_optimizacion_df = pd.concat(
                [
                    resultados_optimizacion_df,
                    pd.DataFrame([error_row])
                ],
                ignore_index=True
            )

            guardar_progreso()


OPTIMIZANDO: Gradient Boosting

----------------------------------------------------------------------
Gradient Boosting | Trial 1/6
Hiperparámetros:
{'model__estimator__subsample': 0.9, 'model__estimator__n_estimators': 150, 'model__estimator__min_samples_leaf': 50, 'model__estimator__max_depth': 3, 'model__estimator__learning_rate': 0.1}
✓ Completado en 22.20 min
MAE=0.2332 | RMSE=0.7245 | R²=0.6102

----------------------------------------------------------------------
Gradient Boosting | Trial 2/6
Hiperparámetros:
{'model__estimator__subsample': 0.7, 'model__estimator__n_estimators': 300, 'model__estimator__min_samples_leaf': 20, 'model__estimator__max_depth': 3, 'model__estimator__learning_rate': 0.08}
✓ Completado en 44.70 min
MAE=0.2354 | RMSE=0.7244 | R²=0.6104

----------------------------------------------------------------------
Gradient Boosting | Trial 3/6
Hiperparámetros:
{'model__estimator__subsample': 0.9, 'model__estimator__n_estimators': 200, 'model__estimator__min_s

## 11. Optimización de CatBoost

CatBoost se procesa de forma independiente porque utiliza directamente las variables categóricas y no necesita el `ColumnTransformer` con One-Hot Encoding.

Se intenta primero la ejecución mediante GPU. Si la versión instalada no admite la configuración multisalida utilizada, se realiza un segundo intento mediante CPU.

In [13]:
# ============================================================
# 11. PREPARACIÓN DE CATBOOST
# ============================================================

X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

for col in categorical_features:

    X_train_cb[col] = (
        X_train_cb[col]
        .fillna("Otros")
        .astype(str)
    )

    X_val_cb[col] = (
        X_val_cb[col]
        .fillna("Otros")
        .astype(str)
    )

cat_features_idx = [
    X_train_cb.columns.get_loc(col)
    for col in categorical_features
]

print("CatBoost preparado.")
print("Variables categóricas:", len(cat_features_idx))

CatBoost preparado.
Variables categóricas: 9


In [14]:
# ============================================================
# 12. OPTIMIZACIÓN CATBOOST
# ============================================================

configuraciones_catboost = list(
    ParameterSampler(
        param_distributions_catboost,
        n_iter=N_ITER_POR_MODELO,
        random_state=RANDOM_STATE
    )
)

for trial_idx, params in enumerate(
    configuraciones_catboost,
    start=1
):

    ya_ejecutado = (
        (len(resultados_optimizacion_df) > 0) and
        (resultados_optimizacion_df["Modelo"] == "CatBoost").any() and
        (resultados_optimizacion_df["Trial"] == trial_idx).any()
    )

    if ya_ejecutado:
        print(
            f"CatBoost Trial {trial_idx}/"
            f"{len(configuraciones_catboost)} ya ejecutado. "
            "Se omite."
        )
        continue

    print("\n" + "=" * 70)
    print(
        f"CATBOOST | Trial "
        f"{trial_idx}/{len(configuraciones_catboost)}"
    )
    print("=" * 70)

    print("Hiperparámetros:")
    print(params)

    inicio = time.time()

    try:

        modelo_catboost = CatBoostRegressor(
            loss_function="MultiRMSE",
            eval_metric="MultiRMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
            task_type="GPU",
            devices="0",
            **params
        )

        try:

            modelo_catboost.fit(
                X_train_cb,
                y_train,
                cat_features=cat_features_idx
            )

            modo = "GPU"

        except Exception as error_gpu:

            print(
                "GPU no disponible para este entrenamiento. "
                "Se intenta CPU."
            )

            modelo_catboost = CatBoostRegressor(
                loss_function="MultiRMSE",
                eval_metric="MultiRMSE",
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
                thread_count=-1,
                **params
            )

            modelo_catboost.fit(
                X_train_cb,
                y_train,
                cat_features=cat_features_idx
            )

            modo = "CPU"

        tiempo_s = time.time() - inicio

        y_pred = modelo_catboost.predict(
            X_val_cb
        )

        y_pred = np.asarray(y_pred)

        detalle, resumen = evaluar_predicciones(
            y_val,
            y_pred,
            "CatBoost",
            trial_idx,
            tiempo_s
        )

        resumen["Hiperparametros"] = json.dumps(
            params,
            ensure_ascii=False,
            default=str
        )

        resumen["Modo"] = modo

        detalle["Hiperparametros"] = resumen["Hiperparametros"]
        detalle["Modo"] = modo

        resultados_optimizacion_df = pd.concat(
            [
                resultados_optimizacion_df,
                pd.DataFrame([resumen])
            ],
            ignore_index=True
        )

        detalle_optimizacion_df = pd.concat(
            [
                detalle_optimizacion_df,
                detalle
            ],
            ignore_index=True
        )

        guardar_progreso()

        print(
            f"✓ Completado en "
            f"{tiempo_s / 60:.2f} min | "
            f"Modo: {modo}"
        )

        print(
            f"MAE={resumen['MAE_Promedio']:.4f} | "
            f"RMSE={resumen['RMSE_Promedio']:.4f} | "
            f"R²={resumen['R2_Promedio']:.4f}"
        )

    except Exception as error:

        tiempo_s = time.time() - inicio

        print(
            f"⚠️ ERROR en CatBoost, trial "
            f"{trial_idx}:"
        )
        print(error)

        error_row = {
            "Modelo": "CatBoost",
            "Trial": trial_idx,
            "MAE_Promedio": np.nan,
            "RMSE_Promedio": np.nan,
            "R2_Promedio": np.nan,
            "Tiempo_entrenamiento_s": tiempo_s,
            "Hiperparametros": json.dumps(
                params,
                ensure_ascii=False,
                default=str
            ),
            "Modo": "ERROR",
            "Estado": f"ERROR: {str(error)}"
        }

        resultados_optimizacion_df = pd.concat(
            [
                resultados_optimizacion_df,
                pd.DataFrame([error_row])
            ],
            ignore_index=True
        )

        guardar_progreso()


CATBOOST | Trial 1/6
Hiperparámetros:
{'random_strength': 0, 'learning_rate': 0.02, 'l2_leaf_reg': 7, 'iterations': 500, 'depth': 6}
✓ Completado en 0.75 min | Modo: GPU
MAE=0.2279 | RMSE=0.7239 | R²=0.6109
CatBoost Trial 2/6 ya ejecutado. Se omite.
CatBoost Trial 3/6 ya ejecutado. Se omite.
CatBoost Trial 4/6 ya ejecutado. Se omite.
CatBoost Trial 5/6 ya ejecutado. Se omite.
CatBoost Trial 6/6 ya ejecutado. Se omite.


In [15]:
# ============================================================
# 13. RESUMEN COMPLETO DE LA OPTIMIZACIÓN
# ============================================================

resultados_validos = resultados_optimizacion_df[
    resultados_optimizacion_df["R2_Promedio"].notna()
].copy()

resultados_validos = (
    resultados_validos
    .sort_values(
        by="R2_Promedio",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("RESULTADOS DE LA OPTIMIZACIÓN")
print("=" * 70)

display(
    resultados_validos[
        [
            "Modelo",
            "Trial",
            "MAE_Promedio",
            "RMSE_Promedio",
            "R2_Promedio",
            "Tiempo_entrenamiento_s"
        ]
    ]
)

RESULTADOS DE LA OPTIMIZACIÓN


,Modelo,Trial,MAE_Promedio,RMSE_Promedio,R2_Promedio,Tiempo_entrenamiento_s
0,CatBoost,1,0.227930,0.723924,0.610906,45.176485
1,Gradient Boosting,2,0.235416,0.724402,0.610365,2682.168955
2,LightGBM,1,0.236816,0.724456,0.610309,25.057624
3,Gradient Boosting,3,0.234429,0.724459,0.610294,1789.226399
4,Gradient Boosting,1,0.233238,0.724539,0.610213,1332.105631
5,Gradient Boosting,4,0.233455,0.724692,0.610063,1429.716607
6,Gradient Boosting,6,0.233031,0.725690,0.608983,1916.243178
7,Gradient Boosting,5,0.233079,0.725917,0.608748,797.090060
8,XGBoost,1,0.239382,0.726856,0.607735,106.829632
9,Extra Trees,1,0.251218,0.747408,0.584844,19910.985416


In [16]:
# ============================================================
# 14. MEJOR CONFIGURACIÓN DE CADA MODELO
# ============================================================

mejores_modelos = (
    resultados_validos
    .sort_values(
        by="R2_Promedio",
        ascending=False
    )
    .groupby(
        "Modelo",
        as_index=False
    )
    .first()
    .sort_values(
        by="R2_Promedio",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("MEJOR CONFIGURACIÓN POR MODELO")
print("=" * 70)

display(
    mejores_modelos[
        [
            "Modelo",
            "Trial",
            "MAE_Promedio",
            "RMSE_Promedio",
            "R2_Promedio",
            "Tiempo_entrenamiento_s",
            "Hiperparametros"
        ]
    ]
)

MEJOR CONFIGURACIÓN POR MODELO


,Modelo,Trial,MAE_Promedio,RMSE_Promedio,R2_Promedio,Tiempo_entrenamiento_s,Hiperparametros
0,CatBoost,1,0.227930,0.723924,0.610906,45.176485,"{""random_strength"": 0, ""learning_rate"": 0.02, ..."
1,Gradient Boosting,2,0.235416,0.724402,0.610365,2682.168955,"{""model__estimator__subsample"": 0.7, ""model__e..."
2,LightGBM,1,0.236816,0.724456,0.610309,25.057624,"{""model__estimator__subsample"": 0.9, ""model__e..."
3,XGBoost,1,0.239382,0.726856,0.607735,106.829632,"{""model__subsample"": 0.7, ""model__n_estimators..."
4,Extra Trees,1,0.251218,0.747408,0.584844,19910.985416,"{""model__n_estimators"": 200, ""model__min_sampl..."


In [17]:
# ============================================================
# 15. EXPORTACIÓN FINAL DE LA OPTIMIZACIÓN
# ============================================================

ruta_mejores = os.path.join(
    carpeta_resultados,
    "Mejores_Hiperparametros_IT.csv"
)

mejores_modelos.to_csv(
    ruta_mejores,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

resultados_optimizacion_df.to_csv(
    ruta_resumen,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

detalle_optimizacion_df.to_csv(
    ruta_detalle,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("ARCHIVOS EXPORTADOS")
print("=" * 70)

print("Resumen completo:")
print(ruta_resumen)

print("Detalle por horizonte:")
print(ruta_detalle)

print("Mejores hiperparámetros:")
print(ruta_mejores)

ARCHIVOS EXPORTADOS
Resumen completo:
c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados_Optimizacion\Resultados_Optimizacion_IT.csv
Detalle por horizonte:
c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados_Optimizacion\Resultados_Optimizacion_IT_Detalle.csv
Mejores hiperparámetros:
c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados_Optimizacion\Mejores_Hiperparametros_IT.csv


In [18]:
# ============================================================
# 16. RESUMEN DE TIEMPOS DE PROCESAMIENTO
# ============================================================

tiempos = (
    resultados_validos
    .groupby("Modelo")
    .agg(
        Trials=("Trial", "count"),
        Tiempo_total_s=("Tiempo_entrenamiento_s", "sum"),
        Tiempo_medio_s=("Tiempo_entrenamiento_s", "mean"),
        Tiempo_minimo_s=("Tiempo_entrenamiento_s", "min"),
        Tiempo_maximo_s=("Tiempo_entrenamiento_s", "max")
    )
    .reset_index()
)

tiempos["Tiempo_total_min"] = (
    tiempos["Tiempo_total_s"] / 60
)

tiempos["Tiempo_medio_min"] = (
    tiempos["Tiempo_medio_s"] / 60
)

tiempos = tiempos.sort_values(
    "Tiempo_total_min",
    ascending=False
)

print("=" * 70)
print("TIEMPO DE PROCESAMIENTO")
print("=" * 70)

display(tiempos)

ruta_tiempos = os.path.join(
    carpeta_resultados,
    "Tiempos_Optimizacion_IT.csv"
)

tiempos.to_csv(
    ruta_tiempos,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo de tiempos:")
print(ruta_tiempos)

TIEMPO DE PROCESAMIENTO


,Modelo,Trials,Tiempo_total_s,Tiempo_medio_s,Tiempo_minimo_s,Tiempo_maximo_s,Tiempo_total_min,Tiempo_medio_min
1,Extra Trees,1,19910.985416,19910.985416,19910.985416,19910.985416,331.849757,331.849757
2,Gradient Boosting,6,9946.550831,1657.758472,797.090060,2682.168955,165.775847,27.629308
4,XGBoost,1,106.829632,106.829632,106.829632,106.829632,1.780494,1.780494
0,CatBoost,1,45.176485,45.176485,45.176485,45.176485,0.752941,0.752941
3,LightGBM,1,25.057624,25.057624,25.057624,25.057624,0.417627,0.417627


Archivo de tiempos:
c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados_Optimizacion\Tiempos_Optimizacion_IT.csv


In [19]:
# ============================================================
# 17. VERIFICACIÓN FINAL
# ============================================================

print("=" * 70)
print("VERIFICACIÓN FINAL DE ARCHIVOS")
print("=" * 70)

archivos = [
    ruta_resumen,
    ruta_detalle,
    ruta_mejores,
    ruta_tiempos
]

for ruta in archivos:

    if os.path.exists(ruta):

        tamaño_kb = os.path.getsize(ruta) / 1024

        print(
            f"✓ {os.path.basename(ruta)} "
            f"({tamaño_kb:.2f} KB)"
        )

    else:

        print(
            f"✗ NO ENCONTRADO: "
            f"{os.path.basename(ruta)}"
        )

print("Test no utilizado durante la optimización.")
print("FIN DEL NOTEBOOK DE OPTIMIZACIÓN.")

VERIFICACIÓN FINAL DE ARCHIVOS
✓ Resultados_Optimizacion_IT.csv (2.94 KB)
✓ Resultados_Optimizacion_IT_Detalle.csv (12.39 KB)
✓ Mejores_Hiperparametros_IT.csv (1.47 KB)
✓ Tiempos_Optimizacion_IT.csv (0.73 KB)
Test no utilizado durante la optimización.
FIN DEL NOTEBOOK DE OPTIMIZACIÓN.


In [20]:
print("KERNEL ACTIVO")

KERNEL ACTIVO
